# Notes & TODO
https://chatgpt.com/c/6aac1700-a894-83eb-bc5d-41ce5c86c37c

# Databricks

# UV MISC

# load_public_data
ner1 - aarhus uni. ccha multiple label dataset; https://huggingface.co/datasets/chcaa/dansk-ner
18 labels
ner2 - ITU ner-name, loc, oorg, other dataset https://sprogteknologi.dk/dataset/danish-named-entity-recognition-data-on-top-of-the-danish-universal-dependencies-data-ud_danish-ddt
4 labels

## pandas
uv pip install pandas fastparquet

In [0]:
# ============================================================
# Download Danish NER datasets -> CSV / pandas
# ============================================================

import os
import json
import pandas as pd
import requests


OUTDIR = "danish_ner_datasets"
os.makedirs(OUTDIR, exist_ok=True)


# ============================================================
# NERset1: chcaa/dansk-ner
# ============================================================

ner1 = {
    "train": pd.read_parquet(
        "hf://datasets/chcaa/dansk-ner/data/train-00000-of-00001.parquet"
    ),
    "dev": pd.read_parquet(
        "hf://datasets/chcaa/dansk-ner/data/dev-00000-of-00001.parquet"
    ),
    "test": pd.read_parquet(
        "hf://datasets/chcaa/dansk-ner/data/test-00000-of-00001.parquet"
    ),
}

print("\n" + "=" * 70)
print("NERset1: chcaa/dansk-ner")
print("=" * 70)

for split, df in ner1.items():

    print(f"\n[{split}]")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))

    display(df.head(3))

    outpath = os.path.join(
        OUTDIR,
        f"NERset1_{split}.csv"
    )

    df.to_csv(
        outpath,
        index=False,
        encoding="utf-8"
    )

    print("Saved:", outpath)


# ============================================================
# NERset2: bplank/danish_ner_transfer
# ============================================================

GH_BASE = (
    "https://raw.githubusercontent.com/"
    "bplank/danish_ner_transfer/master/data"
)


def read_conll(url):

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    rows = []
    tokens = []
    tags = []

    for line in response.text.splitlines():

        if not line.strip():

            if tokens:
                rows.append({
                    "text": " ".join(tokens),
                    "tokens": tokens.copy(),
                    "ner_tags": tags.copy()
                })

                tokens = []
                tags = []

            continue

        token, tag = line.rsplit("\t", 1)

        tokens.append(token)
        tags.append(tag)

    if tokens:
        rows.append({
            "text": " ".join(tokens),
            "tokens": tokens.copy(),
            "ner_tags": tags.copy()
        })

    return pd.DataFrame(rows)


ner2 = {
    "train_5k": read_conll(
        f"{GH_BASE}/da_ddt-ud-ner-train-5k.conll"
    ),
    "train_10k": read_conll(
        f"{GH_BASE}/da_ddt-ud-ner-train-10k.conll"
    ),
    "dev": read_conll(
        f"{GH_BASE}/da_ddt-ud-ner-dev.conll"
    ),
    "test": read_conll(
        f"{GH_BASE}/da_ddt-ud-ner-test.conll"
    ),
}


print("\n" + "=" * 70)
print("NERset2: bplank/danish_ner_transfer")
print("=" * 70)

for split, df in ner2.items():

    print(f"\n[{split}]")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))

    display(df.head(3))

    outpath = os.path.join(
        OUTDIR,
        f"NERset2_{split}.csv"
    )

    df.to_csv(
        outpath,
        index=False,
        encoding="utf-8"
    )

    print("Saved:", outpath)


# ============================================================
# NERset3:
# ai4privacy/pii-masking-openpii-1.5m
#
# IMPORTANT:
# The complete repository is >5 GB.
# Read JSONL as a stream and retain Danish records only.
# ============================================================

HF_BASE = (
    "https://huggingface.co/datasets/"
    "ai4privacy/pii-masking-openpii-1.5m/"
    "resolve/main/data"
)


def read_openpii_danish(url):
    """
    Stream a large OpenPII JSONL file and keep only
    records whose language == 'da'.

    This avoids loading the entire multilingual file into RAM.
    """

    rows = []

    print("Downloading/streaming:")
    print(url)

    with requests.get(
        url,
        stream=True,
        timeout=300
    ) as response:

        response.raise_for_status()

        for i, line in enumerate(
            response.iter_lines(decode_unicode=True),
            start=1
        ):

            if not line:
                continue

            record = json.loads(line)

            if record.get("language") == "da":
                rows.append(record)

            if i % 100000 == 0:
                print(
                    f"Processed {i:,} rows; "
                    f"Danish rows found: {len(rows):,}"
                )

    return pd.DataFrame(rows)


ner3 = {
    "train": read_openpii_danish(
        f"{HF_BASE}/train.jsonl"
    ),
    "validation": read_openpii_danish(
        f"{HF_BASE}/validation.jsonl"
    ),
}


print("\n" + "=" * 70)
print("NERset3: ai4privacy/pii-masking-openpii-1.5m")
print("Filtered to language == 'da'")
print("=" * 70)

for split, df in ner3.items():

    print(f"\n[{split}]")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))

    if len(df) > 0:
        print("\nLanguage counts:")
        print(df["language"].value_counts())

        if "region" in df.columns:
            print("\nRegion counts:")
            print(df["region"].value_counts())

    display(df.head(3))

    # --------------------------------------------------------
    # CSV cannot natively store nested lists/dictionaries.
    # Convert nested OpenPII columns to JSON strings.
    # --------------------------------------------------------

    df_csv = df.copy()

    nested_columns = [
        "privacy_mask",
        "mbert_tokens",
        "mbert_token_classes",
    ]

    for col in nested_columns:
        if col in df_csv.columns:
            df_csv[col] = df_csv[col].apply(
                lambda x: json.dumps(
                    x,
                    ensure_ascii=False
                )
                if x is not None
                else None
            )

    outpath = os.path.join(
        OUTDIR,
        f"NERset3_{split}.csv"
    )

    df_csv.to_csv(
        outpath,
        index=False,
        encoding="utf-8"
    )

    print("Saved:", outpath)


# ============================================================
# NERset3 label summary
# ============================================================

print("\n" + "=" * 70)
print("NERset3 DANISH PII LABEL COUNTS")
print("=" * 70)

for split, df in ner3.items():

    label_counts = {}

    if "privacy_mask" in df.columns:

        for annotations in df["privacy_mask"]:

            if not isinstance(annotations, list):
                continue

            for entity in annotations:

                label = entity.get("label")

                if label is not None:
                    label_counts[label] = (
                        label_counts.get(label, 0) + 1
                    )

    print(f"\n[{split}]")

    for label, count in sorted(
        label_counts.items(),
        key=lambda x: (-x[1], x[0])
    ):
        print(f"{label:25s} {count:,}")


# ============================================================
# List saved files
# ============================================================

print("\n" + "=" * 70)
print("SAVED LOCALLY")
print("=" * 70)

print("Directory:")
print(os.path.abspath(OUTDIR))

for filename in sorted(os.listdir(OUTDIR)):
    print(" -", filename)

## hg_datasets
uv pip install datasets

In [0]:
# ============================================================
# Download Danish NER datasets
# -> Hugging Face Dataset format + JSONL
# ============================================================

import os
import requests

from datasets import (
    load_dataset,
    Dataset,
    DatasetDict,
)


OUTDIR = "danish_ner_datasets_hf"
os.makedirs(OUTDIR, exist_ok=True)


# ============================================================
# NERset1: chcaa/dansk-ner
# ============================================================

ner1 = load_dataset("chcaa/dansk-ner")

print("\n" + "=" * 70)
print("NERset1: chcaa/dansk-ner")
print("=" * 70)

print(ner1)

for split in ner1.keys():

    print(f"\n[{split}]")
    print("Rows:", len(ner1[split]))
    print("Features:")
    print(ner1[split].features)

    display(
        ner1[split]
        .select(range(min(3, len(ner1[split]))))
        .to_pandas()
    )


# Save complete DatasetDict
ner1.save_to_disk(
    os.path.join(OUTDIR, "NERset1")
)


# Save actual JSONL
for split in ner1.keys():

    ner1[split].to_json(
        os.path.join(
            OUTDIR,
            f"NERset1_{split}.jsonl"
        ),
        orient="records",
        lines=True,
        force_ascii=False
    )

    # Preview
    ner1[split].select(
        range(min(20, len(ner1[split])))
    ).to_pandas().to_csv(
        os.path.join(
            OUTDIR,
            f"NERset1_{split}_preview.csv"
        ),
        index=False
    )


# ============================================================
# NERset2: bplank/danish_ner_transfer
# ============================================================

GH_BASE = (
    "https://raw.githubusercontent.com/"
    "bplank/danish_ner_transfer/master/data"
)


def conll_to_dataset(url):

    response = requests.get(
        url,
        timeout=120
    )
    response.raise_for_status()

    rows = []
    tokens = []
    tags = []

    for line in response.text.splitlines():

        line = line.strip()

        if not line:

            if tokens:
                rows.append({
                    "text": " ".join(tokens),
                    "tokens": tokens.copy(),
                    "ner_tags": tags.copy()
                })

                tokens = []
                tags = []

            continue

        token, tag = line.rsplit("\t", 1)

        tokens.append(token)
        tags.append(tag)

    if tokens:
        rows.append({
            "text": " ".join(tokens),
            "tokens": tokens.copy(),
            "ner_tags": tags.copy()
        })

    return Dataset.from_list(rows)


ner2 = DatasetDict({
    "train_5k": conll_to_dataset(
        f"{GH_BASE}/da_ddt-ud-ner-train-5k.conll"
    ),
    "train_10k": conll_to_dataset(
        f"{GH_BASE}/da_ddt-ud-ner-train-10k.conll"
    ),
    "dev": conll_to_dataset(
        f"{GH_BASE}/da_ddt-ud-ner-dev.conll"
    ),
    "test": conll_to_dataset(
        f"{GH_BASE}/da_ddt-ud-ner-test.conll"
    ),
})


print("\n" + "=" * 70)
print("NERset2: bplank/danish_ner_transfer")
print("=" * 70)

print(ner2)

for split in ner2.keys():

    print(f"\n[{split}]")
    print("Rows:", len(ner2[split]))
    print("Features:")
    print(ner2[split].features)

    display(
        ner2[split]
        .select(range(min(3, len(ner2[split]))))
        .to_pandas()
    )


# Save complete DatasetDict
ner2.save_to_disk(
    os.path.join(OUTDIR, "NERset2")
)


# Save JSONL + previews
for split in ner2.keys():

    ner2[split].to_json(
        os.path.join(
            OUTDIR,
            f"NERset2_{split}.jsonl"
        ),
        orient="records",
        lines=True,
        force_ascii=False
    )

    ner2[split].select(
        range(min(20, len(ner2[split])))
    ).to_pandas().to_csv(
        os.path.join(
            OUTDIR,
            f"NERset2_{split}_preview.csv"
        ),
        index=False
    )


# ============================================================
# NERset3:
# ai4privacy/pii-masking-openpii-1.5m
#
# Load in streaming mode, then retain Danish only.
# ============================================================

OPENPII_REPO = (
    "ai4privacy/pii-masking-openpii-1.5m"
)


print("\n" + "=" * 70)
print("NERset3: ai4privacy/pii-masking-openpii-1.5m")
print("Streaming + filtering language == 'da'")
print("=" * 70)


openpii_stream = load_dataset(
    OPENPII_REPO,
    streaming=True
)


def materialize_danish(streaming_dataset, split_name):
    """
    Iterate over one OpenPII split and materialize
    only Danish records into an ordinary Dataset.
    """

    rows = []

    for i, example in enumerate(
        streaming_dataset,
        start=1
    ):

        if example.get("language") == "da":
            rows.append(example)

        if i % 100000 == 0:
            print(
                f"{split_name}: "
                f"processed {i:,}; "
                f"Danish found {len(rows):,}"
            )

    print(
        f"{split_name}: finished. "
        f"Danish rows = {len(rows):,}"
    )

    return Dataset.from_list(rows)


ner3 = DatasetDict({
    split: materialize_danish(
        openpii_stream[split],
        split
    )
    for split in openpii_stream.keys()
})


print("\n" + "=" * 70)
print("NERset3 DANISH DATA")
print("=" * 70)

print(ner3)


for split in ner3.keys():

    print(f"\n[{split}]")
    print("Rows:", len(ner3[split]))

    print("\nFeatures:")
    print(ner3[split].features)

    if len(ner3[split]) > 0:

        display(
            ner3[split]
            .select(
                range(
                    min(3, len(ner3[split]))
                )
            )
            .to_pandas()
        )


# ============================================================
# Save complete Danish OpenPII DatasetDict
# ============================================================

ner3.save_to_disk(
    os.path.join(
        OUTDIR,
        "NERset3_OpenPII_Danish"
    )
)


# ============================================================
# Save actual JSONL files
# ============================================================

for split in ner3.keys():

    json_path = os.path.join(
        OUTDIR,
        f"NERset3_{split}.jsonl"
    )

    ner3[split].to_json(
        json_path,
        orient="records",
        lines=True,
        force_ascii=False
    )

    print("Saved:", json_path)


    # CSV preview only
    preview_path = os.path.join(
        OUTDIR,
        f"NERset3_{split}_preview.csv"
    )

    ner3[split].select(
        range(
            min(20, len(ner3[split]))
        )
    ).to_pandas().to_csv(
        preview_path,
        index=False
    )


# ============================================================
# Show PII label distribution for Danish OpenPII
# ============================================================

print("\n" + "=" * 70)
print("NERset3 DANISH PII LABEL COUNTS")
print("=" * 70)


for split in ner3.keys():

    label_counts = {}

    for example in ner3[split]:

        for entity in example.get(
            "privacy_mask", []
        ):

            label = entity.get("label")

            if label:
                label_counts[label] = (
                    label_counts.get(label, 0)
                    + 1
                )

    print(f"\n[{split}]")

    for label, count in sorted(
        label_counts.items(),
        key=lambda x: (-x[1], x[0])
    ):
        print(
            f"{label:25s} "
            f"{count:,}"
        )


# ============================================================
# Show saved files/directories
# ============================================================

print("\n" + "=" * 70)
print("SAVED UNDER:")
print(os.path.abspath(OUTDIR))
print("=" * 70)

for root, dirs, files in os.walk(OUTDIR):

    level = (
        root.replace(OUTDIR, "")
        .count(os.sep)
    )

    indent = "  " * level

    print(
        f"{indent}"
        f"{os.path.basename(root)}/"
    )

    for filename in files:
        print(
            f"{indent}  {filename}"
        )

# preproc

## hg_data_ALFalign

In [0]:

from __future__ import annotations

import json
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple


# ============================================================
# Paths
# ============================================================

DEFAULT_INPUT_DIR = Path("danish_ner_datasets_hf")
DEFAULT_OUTPUT_DIR = Path("danish_ner_datasets_aligned")
DEFAULT_FAIL_ON_UNKNOWN_LABEL = True


# ============================================================
# ALF child-label inventory
#
# Note:
# Several ALF parent labels are ALSO valid child labels
# (e.g. PER, LOC, ORG, CONTACT, ID, DATE, DEM, HEALTH, NUM).
# MISC has no child label, so no source label is projected to MISC.
# ============================================================

ALF_CHILD_LABELS = {
    "PER", "STAFF", "PATIENT",
    "LOC", "ADDRESS", "POSTCODE",
    "ORG", "HOSPITAL",
    "CONTACT", "PHONE", "EMAIL",
    "ID", "CPR",
    "DATE", "TIME", "DURATION",
    "DEM", "ETHNICITY", "RELIGION", "POLITICS", "SEXUALITY", "AGE",
    "HEALTH", "DIAGNOSIS", "MEDICATION", "CONDITION",
    "REL",
    "NUM", "PERCENT", "CARDINAL", "MONEY",
}


# ============================================================
# EXPLICIT SOURCE -> ALF CHILD-LABEL PROJECTIONS
#
# None means:
#   "do not create an ALF child-label entity from this source tag".
#
# The mapping is intentionally conservative where the source tag
# is broader/different than the ALF PII child taxonomy.
# ============================================================

LABEL_PROJECTIONS: Dict[str, Dict[str, Optional[str]]] = {

    # --------------------------------------------------------
    # NERset1: chcaa/dansk-ner
    # DANSK fine-grained NER labels
    # --------------------------------------------------------
    "NERset1": {
        "PERSON": "PER",
        "GPE": "LOC",
        "LOCATION": "LOC",
        "FACILITY": "LOC",
        "ORGANIZATION": "ORG",

        "DATE": "DATE",
        "TIME": "TIME",

        "CARDINAL": "CARDINAL",
        "PERCENT": "PERCENT",
        "MONEY": "MONEY",
        "QUANTITY": "NUM",
        "ORDINAL": "NUM",

        # NORP = nationalities / religious / political groups.
        # The source tag does not distinguish ETHNICITY,
        # RELIGION, and POLITICS, so use the broader ALF child
        # fallback DEM.
        "NORP": "DEM",

        # Generic language mentions are not necessarily a
        # demographic attribute of a person ("native language"),
        # so they are not projected automatically.
        "LANGUAGE": None,

        # No confident ALF child-label equivalent.
        "EVENT": None,
        "LAW": None,
        "PRODUCT": None,
        "WORK OF ART": None,

        # Alias included defensively in case an exported version
        # uses underscores.
        "WORK_OF_ART": None,
    },

    # --------------------------------------------------------
    # NERset2: bplank/danish_ner_transfer
    # CoNLL labels: PER / LOC / ORG / MISC
    # --------------------------------------------------------
    "NERset2": {
        "PER": "PER",
        "LOC": "LOC",
        "ORG": "ORG",

        # ALF MISC has no child label.
        "MISC": None,
    },

    # --------------------------------------------------------
    # NERset3: ai4privacy/pii-masking-openpii-1.5m
    # 19-label OpenPII taxonomy
    # --------------------------------------------------------
    "NERset3": {
        "GIVENNAME": "PER",
        "SURNAME": "PER",

        "CITY": "LOC",
        "STREET": "ADDRESS",
        "BUILDINGNUM": "ADDRESS",
        "ZIPCODE": "POSTCODE",

        "EMAIL": "EMAIL",
        "TELEPHONENUM": "PHONE",

        "IDCARDNUM": "ID",
        "DRIVERLICENSENUM": "ID",
        "TAXNUM": "ID",
        "SOCIALNUM": "ID",
        "PASSPORTNUM": "ID",

        # ALF has no separate financial-account-number child
        # tag. Treat a credit-card number as an identifier under
        # the broad ALF ID child label.
        "CREDITCARDNUMBER": "ID",

        "DATE": "DATE",
        "AGE": "AGE",

        # ALF has no TITLE/GENDER/SEX child tags. DEM is the
        # child-level fallback for demographic/person attributes.
        "TITLE": "DEM",
        "GENDER": "DEM",
        "SEX": "DEM",
    },
}


# ============================================================
# Validate projection dictionary
# ============================================================

for dataset_name, projection in LABEL_PROJECTIONS.items():
    for source_label, child_label in projection.items():
        if child_label is not None and child_label not in ALF_CHILD_LABELS:
            raise ValueError(
                f"{dataset_name}: invalid ALF child label "
                f"{child_label!r} for source label {source_label!r}"
            )


# ============================================================
# Audit counters
# ============================================================

mapped_counts: Dict[str, Counter] = defaultdict(Counter)
intentionally_unmapped_counts: Dict[str, Counter] = defaultdict(Counter)
unknown_counts: Dict[str, Counter] = defaultdict(Counter)


def project_source_label(
    dataset_name: str,
    source_label: str,
    *,
    count: bool = True,
) -> Optional[str]:
    """
    Project one source label to an ALF child label.

    Returns:
        ALF child label, or None if intentionally unmapped /
        unknown.
    """
    source_label = str(source_label).strip()
    projection = LABEL_PROJECTIONS[dataset_name]

    if source_label not in projection:
        if count:
            unknown_counts[dataset_name][source_label] += 1
        return None

    child_label = projection[source_label]

    if count:
        if child_label is None:
            intentionally_unmapped_counts[dataset_name][source_label] += 1
        else:
            mapped_counts[dataset_name][
                f"{source_label} -> {child_label}"
            ] += 1

    return child_label


def project_bio_tag(
    dataset_name: str,
    tag: str,
) -> str:
    """
    Project a BIO tag such as B-PER or I-GIVENNAME.

    Unmapped source labels become O.
    """
    if tag is None:
        return "O"

    tag = str(tag).strip()

    if tag == "" or tag == "O":
        return "O"

    if "-" not in tag:
        # Unexpected token label. Record it as unknown.
        unknown_counts[dataset_name][tag] += 1
        return "O"

    prefix, source_label = tag.split("-", 1)
    child_label = project_source_label(
        dataset_name,
        source_label,
        count=False,
    )

    if source_label not in LABEL_PROJECTIONS[dataset_name]:
        unknown_counts[dataset_name][source_label] += 1
        return "O"

    if child_label is None:
        return "O"

    return f"{prefix}-{child_label}"


# ============================================================
# Generic helpers
# ============================================================

def make_alf_entity(
    text: str,
    start: int,
    end: int,
    source_label: str,
    child_label: str,
) -> Dict[str, Any]:
    start = int(start)
    end = int(end)

    return {
        "start": start,
        "end": end,
        "text": text[start:end],
        "source_label": source_label,
        "child_label": child_label,
    }


def token_offsets_for_joined_text(
    tokens: List[str],
) -> List[Tuple[int, int]]:
    """
    NERset2 text was created by:
        " ".join(tokens)

    Therefore the character offsets are deterministic.
    """
    offsets: List[Tuple[int, int]] = []
    cursor = 0

    for token in tokens:
        start = cursor
        end = start + len(token)
        offsets.append((start, end))
        cursor = end + 1  # one joining space

    return offsets


def source_entities_from_bio(
    tokens: List[str],
    tags: List[str],
    text: str,
) -> List[Dict[str, Any]]:
    """
    Convert token-level BIO annotations to source char spans.

    Handles malformed I-X by starting a new entity.
    """
    if len(tokens) != len(tags):
        raise ValueError(
            f"tokens/tags length mismatch: {len(tokens)} vs {len(tags)}"
        )

    offsets = token_offsets_for_joined_text(tokens)
    entities: List[Dict[str, Any]] = []
    current: Optional[Dict[str, Any]] = None

    def flush() -> None:
        nonlocal current
        if current is not None:
            current["text"] = text[current["start"]:current["end"]]
            entities.append(current)
            current = None

    for token, tag, (start, end) in zip(tokens, tags, offsets):
        tag = str(tag).strip()

        if tag == "O" or tag == "":
            flush()
            continue

        if "-" not in tag:
            flush()
            # Keep malformed label visible to the audit.
            entities.append({
                "start": start,
                "end": end,
                "text": text[start:end],
                "source_label": tag,
            })
            continue

        prefix, source_label = tag.split("-", 1)

        if (
            prefix == "I"
            and current is not None
            and current["source_label"] == source_label
        ):
            current["end"] = end
        else:
            flush()
            current = {
                "start": start,
                "end": end,
                "source_label": source_label,
            }

    flush()
    return entities


def char_spans_to_token_bio(
    token_dicts: List[Dict[str, Any]],
    alf_entities: List[Dict[str, Any]],
) -> List[str]:
    """
    Create token BIO labels for NERset1 using its token char offsets.
    """
    result: List[str] = []
    previous_entity_index: Optional[int] = None

    sorted_entities = sorted(
        enumerate(alf_entities),
        key=lambda x: (x[1]["start"], x[1]["end"]),
    )

    for token in token_dicts:
        token_start = int(token["start"])
        token_end = int(token["end"])

        matched_index: Optional[int] = None
        matched_entity: Optional[Dict[str, Any]] = None

        for entity_index, entity in sorted_entities:
            # Any character overlap.
            if (
                token_start < entity["end"]
                and token_end > entity["start"]
            ):
                matched_index = entity_index
                matched_entity = entity
                break

        if matched_entity is None:
            result.append("O")
            previous_entity_index = None
            continue

        prefix = (
            "I"
            if previous_entity_index == matched_index
            else "B"
        )

        result.append(
            f"{prefix}-{matched_entity['child_label']}"
        )
        previous_entity_index = matched_index

    return result


# ============================================================
# Dataset-specific alignment
# ============================================================

def align_nerset1(
    record: Dict[str, Any],
    split: str,
) -> Dict[str, Any]:
    """
    chcaa/dansk-ner:
        text
        ents = [{start, end, label}, ...]
    """
    out = dict(record)
    text = str(record.get("text", ""))

    alf_entities: List[Dict[str, Any]] = []

    for entity in record.get("ents", []) or []:
        source_label = str(entity["label"]).strip()
        child_label = project_source_label(
            "NERset1",
            source_label,
        )

        if child_label is None:
            continue

        alf_entities.append(
            make_alf_entity(
                text=text,
                start=entity["start"],
                end=entity["end"],
                source_label=source_label,
                child_label=child_label,
            )
        )

    out["alf_dataset"] = "NERset1"
    out["alf_split"] = split
    out["alf_entities"] = alf_entities

    tokens = record.get("tokens")
    if (
        isinstance(tokens, list)
        and all(
            isinstance(x, dict)
            and "start" in x
            and "end" in x
            for x in tokens
        )
    ):
        out["alf_ner_tags"] = char_spans_to_token_bio(
            tokens,
            alf_entities,
        )

    return out


def align_nerset2(
    record: Dict[str, Any],
    split: str,
) -> Dict[str, Any]:
    """
    bplank/danish_ner_transfer:
        text
        tokens
        ner_tags  (BIO strings)
    """
    out = dict(record)

    text = str(record.get("text", ""))
    tokens = [str(x) for x in record.get("tokens", [])]
    tags = [str(x) for x in record.get("ner_tags", [])]

    source_entities = source_entities_from_bio(
        tokens=tokens,
        tags=tags,
        text=text,
    )

    alf_entities: List[Dict[str, Any]] = []

    for entity in source_entities:
        source_label = str(entity["source_label"]).strip()
        child_label = project_source_label(
            "NERset2",
            source_label,
        )

        if child_label is None:
            continue

        alf_entities.append(
            make_alf_entity(
                text=text,
                start=entity["start"],
                end=entity["end"],
                source_label=source_label,
                child_label=child_label,
            )
        )

    out["alf_dataset"] = "NERset2"
    out["alf_split"] = split
    out["alf_entities"] = alf_entities

    # Directly usable token-level ALF BIO labels.
    out["alf_ner_tags"] = [
        project_bio_tag("NERset2", tag)
        for tag in tags
    ]

    return out


def align_nerset3(
    record: Dict[str, Any],
    split: str,
) -> Dict[str, Any]:
    """
    ai4privacy/pii-masking-openpii-1.5m:
        source_text
        privacy_mask = [{start, end, label, ...}, ...]
        mbert_token_classes = BIO labels
    """
    out = dict(record)
    text = str(record.get("source_text", ""))

    # Add a common text field without removing source_text.
    out["text"] = text

    alf_entities: List[Dict[str, Any]] = []

    for entity in record.get("privacy_mask", []) or []:
        source_label = str(entity["label"]).strip()
        child_label = project_source_label(
            "NERset3",
            source_label,
        )

        if child_label is None:
            continue

        alf_entities.append(
            make_alf_entity(
                text=text,
                start=entity["start"],
                end=entity["end"],
                source_label=source_label,
                child_label=child_label,
            )
        )

    out["alf_dataset"] = "NERset3"
    out["alf_split"] = split
    out["alf_entities"] = alf_entities

    source_token_tags = record.get("mbert_token_classes")
    if isinstance(source_token_tags, list):
        out["alf_mbert_token_classes"] = [
            project_bio_tag("NERset3", tag)
            for tag in source_token_tags
        ]

    return out


# ============================================================
# JSONL I/O
# ============================================================

def split_from_filename(
    path: Path,
    dataset_name: str,
) -> str:
    prefix = f"{dataset_name}_"
    stem = path.stem

    if not stem.startswith(prefix):
        return "unknown"

    return stem[len(prefix):]


def iter_jsonl(path: Path) -> Iterable[Dict[str, Any]]:
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                yield json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(
                    f"{path}:{line_number}: invalid JSON"
                ) from e


def write_aligned_file(
    input_path: Path,
    dataset_name: str,
    output_dir: Path,
) -> Tuple[Path, int]:
    split = split_from_filename(
        input_path,
        dataset_name,
    )

    if dataset_name == "NERset1":
        aligner = align_nerset1
    elif dataset_name == "NERset2":
        aligner = align_nerset2
    elif dataset_name == "NERset3":
        aligner = align_nerset3
    else:
        raise ValueError(
            f"Unsupported dataset: {dataset_name}"
        )

    output_path = output_dir / (
        f"{input_path.stem}_aligned.jsonl"
    )

    n_rows = 0

    with output_path.open(
        "w",
        encoding="utf-8",
    ) as fout:

        for record in iter_jsonl(input_path):
            aligned = aligner(record, split)

            fout.write(
                json.dumps(
                    aligned,
                    ensure_ascii=False,
                )
                + "\n"
            )

            n_rows += 1

    return output_path, n_rows


# ============================================================
# Public runner / Main
# ============================================================

def reset_audit_counters() -> None:
    """Reset module-level audit counters before each complete run."""
    mapped_counts.clear()
    intentionally_unmapped_counts.clear()
    unknown_counts.clear()


def align_danish_ner_to_alf(
    input_dir: str | Path = DEFAULT_INPUT_DIR,
    output_dir: str | Path = DEFAULT_OUTPUT_DIR,
    *,
    fail_on_unknown_label: bool = DEFAULT_FAIL_ON_UNKNOWN_LABEL,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Align all downloaded Danish NER JSONL files to ALF child labels.

    This is the recommended entry point for both Python scripts and
    Jupyter notebooks. It can safely be called multiple times in the
    same kernel because audit counters are reset at the start of each run.

    Parameters
    ----------
    input_dir:
        Folder containing NERset1_*.jsonl, NERset2_*.jsonl,
        and NERset3_*.jsonl.
    output_dir:
        Folder where aligned JSONL files and audit JSON files are written.
    fail_on_unknown_label:
        If True, raise RuntimeError after writing the audit summary when
        source labels not explicitly present in LABEL_PROJECTIONS are seen.
    verbose:
        Print progress information.

    Returns
    -------
    dict
        The same audit summary that is written to alignment_summary.json,
        plus paths to the projection and summary files.
    """
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    reset_audit_counters()

    if not input_dir.exists():
        raise FileNotFoundError(
            f"Input folder does not exist: {input_dir.resolve()}"
        )

    file_groups = {
        "NERset1": sorted(input_dir.glob("NERset1_*.jsonl")),
        "NERset2": sorted(input_dir.glob("NERset2_*.jsonl")),
        "NERset3": sorted(input_dir.glob("NERset3_*.jsonl")),
    }

    if not any(file_groups.values()):
        raise FileNotFoundError(
            f"No NERset*.jsonl files found under {input_dir.resolve()}"
        )

    if verbose:
        print("=" * 80)
        print("ALF PII CHILD-LABEL PROJECTION")
        print("=" * 80)
        print("Input :", input_dir.resolve())
        print("Output:", output_dir.resolve())

    processed_files = []

    for dataset_name, paths in file_groups.items():
        if verbose:
            print(f"\n[{dataset_name}]")

        if not paths:
            if verbose:
                print("  No JSONL files found.")
            continue

        for input_path in paths:
            output_path, n_rows = write_aligned_file(
                input_path,
                dataset_name,
                output_dir,
            )

            processed_files.append({
                "dataset": dataset_name,
                "input": str(input_path),
                "output": str(output_path),
                "rows": n_rows,
            })

            if verbose:
                print(
                    f"  {input_path.name} "
                    f"-> {output_path.name} "
                    f"({n_rows:,} rows)"
                )

    # Save the explicit projection dictionary.
    projection_path = output_dir / "alf_child_label_projection.json"
    with projection_path.open("w", encoding="utf-8") as f:
        json.dump(
            LABEL_PROJECTIONS,
            f,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )

    summary: Dict[str, Any] = {
        "processed_files": processed_files,
        "mapped_entity_counts": {
            dataset: dict(counter)
            for dataset, counter in mapped_counts.items()
        },
        "intentionally_unmapped_entity_counts": {
            dataset: dict(counter)
            for dataset, counter in intentionally_unmapped_counts.items()
        },
        "unknown_source_label_counts": {
            dataset: dict(counter)
            for dataset, counter in unknown_counts.items()
        },
    }

    summary_path = output_dir / "alignment_summary.json"
    with summary_path.open("w", encoding="utf-8") as f:
        json.dump(
            summary,
            f,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )

    # Include convenient return-only metadata for notebook callers.
    result = dict(summary)
    result["projection_file"] = str(projection_path)
    result["summary_file"] = str(summary_path)
    result["output_dir"] = str(output_dir)

    if verbose:
        print("\n" + "=" * 80)
        print("PROJECTION FILE:", projection_path)
        print("SUMMARY FILE   :", summary_path)
        print("=" * 80)

    unknown_nonempty = {
        dataset: dict(counter)
        for dataset, counter in unknown_counts.items()
        if counter
    }

    if unknown_nonempty:
        if verbose:
            print("\nUnknown source labels detected:")
            print(
                json.dumps(
                    unknown_nonempty,
                    ensure_ascii=False,
                    indent=2,
                )
            )

        if fail_on_unknown_label:
            raise RuntimeError(
                "Unknown source labels were found. "
                "Update LABEL_PROJECTIONS explicitly. "
                f"See {summary_path}."
            )

    if verbose:
        print("\nFinished successfully.")

    return result


def main(
    input_dir: str | Path = DEFAULT_INPUT_DIR,
    output_dir: str | Path = DEFAULT_OUTPUT_DIR,
    *,
    fail_on_unknown_label: bool = DEFAULT_FAIL_ON_UNKNOWN_LABEL,
    verbose: bool = True,
) -> Dict[str, Any]:
    """Backward-compatible wrapper around align_danish_ner_to_alf()."""
    return align_danish_ner_to_alf(
        input_dir=input_dir,
        output_dir=output_dir,
        fail_on_unknown_label=fail_on_unknown_label,
        verbose=verbose,
    )


if __name__ == "__main__":
    main()


## en_tag_2_dk

In [0]:
from __future__ import annotations

import json
from copy import deepcopy
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Union


# ============================================================
# Explicit ALF English -> Danish child-label projection
#
# The dictionary is intentionally explicit so that the mapping
# can be reviewed/edited without changing the conversion logic.
# ============================================================

EN_TO_DK_LABELS: Dict[str, str] = {
    "PER": "PERSON",
    "STAFF": "PERSONALE",
    "PATIENT": "PATIENT",

    "LOC": "STED",
    "ADDRESS": "ADRESSE",
    "POSTCODE": "POSTNUMMER",

    "ORG": "ORGANISATION",
    "HOSPITAL": "HOSPITAL",

    "CONTACT": "KONTAKT",
    "PHONE": "TELEFON",
    "EMAIL": "EMAIL",

    "ID": "ID",
    "CPR": "CPR",

    "DATE": "DATO",
    "TIME": "TID",
    "DURATION": "VARIGHED",

    "DEM": "DEMOGRAFI",
    "ETHNICITY": "ETNICITET",
    "RELIGION": "RELIGION",
    "POLITICS": "POLITIK",
    "SEXUALITY": "SEKSUALITET",
    "AGE": "ALDER",

    "HEALTH": "SUNDHED",
    "DIAGNOSIS": "DIAGNOSE",
    "MEDICATION": "MEDICIN",
    "CONDITION": "TILSTAND",

    "REL": "RELATION",

    "NUM": "TAL",
    "PERCENT": "PROCENT",
    "CARDINAL": "KARDINALTAL",
    "MONEY": "BELØB",
}


# Fields produced by the previous ALF-alignment script.
BIO_FIELDS = {
    "alf_ner_tags",
    "alf_mbert_token_classes",
}


# ============================================================
# Label conversion helpers
# ============================================================

def project_label_en_to_dk(
    label: Optional[str],
    *,
    strict: bool = True,
) -> Optional[str]:
    """
    Convert one ALF child label from English to Danish.

    Examples
    --------
    PER -> PERSON
    ADDRESS -> ADRESSE
    CONDITION -> TILSTAND

    None is returned unchanged.
    """
    if label is None:
        return None

    label = str(label).strip()

    if label in EN_TO_DK_LABELS:
        return EN_TO_DK_LABELS[label]

    if strict:
        raise KeyError(
            f"Unknown ALF English label: {label!r}. "
            "Add it explicitly to EN_TO_DK_LABELS."
        )

    return label


def project_bio_tag_en_to_dk(
    tag: Optional[str],
    *,
    strict: bool = True,
) -> str:
    """
    Convert a BIO/BIOES-style ALF tag while preserving the prefix.

    Examples
    --------
    B-PER       -> B-PERSON
    I-ADDRESS   -> I-ADRESSE
    O           -> O
    """
    if tag is None:
        return "O"

    tag = str(tag).strip()

    if tag == "" or tag == "O":
        return "O"

    if "-" not in tag:
        # Defensive support for a plain ALF label.
        projected = project_label_en_to_dk(tag, strict=strict)
        return str(projected)

    prefix, label = tag.split("-", 1)
    projected = project_label_en_to_dk(label, strict=strict)
    return f"{prefix}-{projected}"


# ============================================================
# Record conversion
# ============================================================

def convert_record_en_to_dk(
    record: Dict[str, Any],
    *,
    keep_english: bool = True,
    strict: bool = True,
) -> Dict[str, Any]:
    """
    Convert ALF labels in one record produced by the previous
    alignment script.

    Converted fields
    ----------------
    1. alf_entities[*].child_label
    2. alf_ner_tags
    3. alf_mbert_token_classes

    Source-taxonomy fields such as source_label, ents.label,
    ner_tags, privacy_mask.label are intentionally NOT changed,
    because they describe the original source datasets.

    If keep_english=True, the original ALF English labels are
    retained in parallel *_en fields before replacement.
    """
    out = deepcopy(record)

    # --------------------------------------------------------
    # Entity-level labels
    # --------------------------------------------------------
    entities = out.get("alf_entities")

    if isinstance(entities, list):
        for entity in entities:
            if not isinstance(entity, dict):
                continue

            if "child_label" not in entity:
                continue

            en_label = entity.get("child_label")

            if keep_english and "child_label_en" not in entity:
                entity["child_label_en"] = en_label

            entity["child_label"] = project_label_en_to_dk(
                en_label,
                strict=strict,
            )

    # --------------------------------------------------------
    # Token-level ALF labels
    # --------------------------------------------------------
    for field in BIO_FIELDS:
        tags = out.get(field)

        if not isinstance(tags, list):
            continue

        if keep_english:
            en_field = f"{field}_en"
            if en_field not in out:
                out[en_field] = deepcopy(tags)

        out[field] = [
            project_bio_tag_en_to_dk(tag, strict=strict)
            for tag in tags
        ]

    out["alf_label_language"] = "da"

    return out


# ============================================================
# JSONL / JSONDK I/O
#
# .jsondk is still newline-delimited JSON. Only the extension
# changes, as requested.
# ============================================================

def iter_jsonl(path: Union[str, Path]) -> Iterable[Dict[str, Any]]:
    path = Path(path)

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"{path}:{line_number}: invalid JSON"
                ) from exc

            if not isinstance(obj, dict):
                raise ValueError(
                    f"{path}:{line_number}: expected a JSON object, "
                    f"got {type(obj).__name__}"
                )

            yield obj


def default_jsondk_path(
    input_path: Union[str, Path],
    output_dir: Optional[Union[str, Path]] = None,
) -> Path:
    """
    For an aligned input such as:
        NERset2_dev_aligned.jsonl

    create:
        NERset2_dev_aligned.jsondk
    """
    input_path = Path(input_path)

    if input_path.suffix.lower() != ".jsonl":
        raise ValueError(
            f"Expected a .jsonl input file, got: {input_path}"
        )

    filename = input_path.with_suffix(".jsondk").name

    if output_dir is None:
        return input_path.with_suffix(".jsondk")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir / filename


def convert_jsonl_to_jsondk(
    input_path: Union[str, Path],
    output_path: Optional[Union[str, Path]] = None,
    *,
    output_dir: Optional[Union[str, Path]] = None,
    keep_english: bool = True,
    strict: bool = True,
) -> Dict[str, Any]:
    """
    Convert one aligned .jsonl file to a .jsondk file.

    Parameters
    ----------
    input_path:
        Input .jsonl produced by the previous ALF alignment.

    output_path:
        Optional explicit output filename. If omitted, the input
        suffix .jsonl is replaced by .jsondk.

    output_dir:
        Optional destination directory used only when output_path
        is not provided.

    keep_english:
        Preserve English ALF labels in *_en fields.

    strict:
        If True, stop on any ALF label not explicitly present in
        EN_TO_DK_LABELS. This prevents silent label corruption.

    Returns
    -------
    dict with paths and record count.
    """
    input_path = Path(input_path)

    if input_path.suffix.lower() != ".jsonl":
        raise ValueError(
            f"Expected input extension .jsonl, got: {input_path}"
        )

    if not input_path.exists():
        raise FileNotFoundError(input_path)

    if output_path is not None and output_dir is not None:
        raise ValueError(
            "Use either output_path or output_dir, not both."
        )

    if output_path is None:
        output_path = default_jsondk_path(
            input_path,
            output_dir=output_dir,
        )
    else:
        output_path = Path(output_path)
        if output_path.suffix.lower() != ".jsondk":
            output_path = output_path.with_suffix(".jsondk")
        output_path.parent.mkdir(parents=True, exist_ok=True)

    n_records = 0

    with Path(output_path).open("w", encoding="utf-8") as fout:
        for record in iter_jsonl(input_path):
            converted = convert_record_en_to_dk(
                record,
                keep_english=keep_english,
                strict=strict,
            )

            fout.write(
                json.dumps(
                    converted,
                    ensure_ascii=False,
                    separators=(",", ":"),
                )
                + "\n"
            )
            n_records += 1

    return {
        "input": str(input_path),
        "output": str(output_path),
        "records": n_records,
        "keep_english": keep_english,
        "strict": strict,
    }


# ============================================================
# Batch conversion for a folder of aligned JSONL files
# ============================================================

def convert_aligned_folder_to_jsondk(
    input_dir: Union[str, Path] = "danish_ner_datasets_aligned",
    output_dir: Optional[Union[str, Path]] = None,
    *,
    recursive: bool = False,
    keep_english: bool = True,
    strict: bool = True,
) -> Dict[str, Any]:
    """
    Convert every .jsonl in an aligned-data folder to .jsondk.

    By default the .jsondk files are written beside their input
    .jsonl files. Pass output_dir to place them elsewhere.
    """
    input_dir = Path(input_dir)

    if not input_dir.exists():
        raise FileNotFoundError(input_dir)

    pattern = "**/*.jsonl" if recursive else "*.jsonl"
    input_files = sorted(input_dir.glob(pattern))

    if not input_files:
        raise FileNotFoundError(
            f"No .jsonl files found under {input_dir}"
        )

    results: List[Dict[str, Any]] = []

    for input_path in input_files:
        if output_dir is None:
            this_output_dir = None
        else:
            # Preserve relative subdirectories when recursive=True.
            rel_parent = input_path.parent.relative_to(input_dir)
            this_output_dir = Path(output_dir) / rel_parent

        result = convert_jsonl_to_jsondk(
            input_path,
            output_dir=this_output_dir,
            keep_english=keep_english,
            strict=strict,
        )
        results.append(result)

    return {
        "input_dir": str(input_dir),
        "output_dir": (
            str(output_dir)
            if output_dir is not None
            else str(input_dir)
        ),
        "files": results,
        "n_files": len(results),
        "n_records": sum(x["records"] for x in results),
    }


# ============================================================
# Convenient Jupyter entry point
# ============================================================

def main(
    input_path: Union[str, Path] = "danish_ner_datasets_aligned",
    output_dir: Optional[Union[str, Path]] = None,
    *,
    keep_english: bool = True,
    strict: bool = True,
) -> Dict[str, Any]:
    """
    Jupyter-friendly entry point.

    - If input_path is a .jsonl file: convert one file.
    - If input_path is a directory: convert all .jsonl files.
    """
    input_path = Path(input_path)

    if input_path.is_file():
        return convert_jsonl_to_jsondk(
            input_path,
            output_dir=output_dir,
            keep_english=keep_english,
            strict=strict,
        )

    if input_path.is_dir():
        return convert_aligned_folder_to_jsondk(
            input_dir=input_path,
            output_dir=output_dir,
            keep_english=keep_english,
            strict=strict,
        )

    raise FileNotFoundError(input_path)


if __name__ == "__main__":
    result = main()

    print(json.dumps(
        result,
        ensure_ascii=False,
        indent=2,
    ))


## Format to [PersonData]..[Label]

In [0]:
from __future__ import annotations

import json
from copy import deepcopy
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple, Union


# ============================================================
# USER-ADJUSTABLE FORMAT SETTINGS
# ============================================================

# Inline PII format.
#
# Default:
#     [PersonData]Peter Hansen[PERSON]
#
# Available placeholders:
#     {text}   = original entity text
#     {label}  = Danish ALF child label
#
# Examples:
#     INLINE_ENTITY_FORMAT = "[PersonData]{text}[{label}]"
#     INLINE_ENTITY_FORMAT = "<PII>{text}</{label}>"
#     INLINE_ENTITY_FORMAT = "[PII:{label}]{text}[/PII]"
#
INLINE_ENTITY_FORMAT = "[PersonData]{text}[{label}]"

# Field names expected in the .jsondk input.
TEXT_FIELD = "text"
ENTITIES_FIELD = "alf_entities"
ENTITY_START_FIELD = "start"
ENTITY_END_FIELD = "end"
ENTITY_TEXT_FIELD = "text"
ENTITY_LABEL_FIELD = "child_label"

# Name of the newly generated inline-formatted text field.
OUTPUT_TEXT_FIELD = "alf_text"

# Keep the original record and add OUTPUT_TEXT_FIELD.
# If False, output records contain only OUTPUT_TEXT_FIELD.
KEEP_ORIGINAL_FIELDS = True

# Validate that entity["text"] matches text[start:end].
VALIDATE_ENTITY_TEXT = True

# What to do when spans overlap.
#
# "error"      -> raise an exception (safest/default)
# "skip_later" -> keep the first span and skip later overlaps
# "longest"    -> prefer the longest span among overlapping spans
OVERLAP_POLICY = "error"

# Empty / None labels:
# "error" -> stop
# "skip"  -> leave that entity unformatted
EMPTY_LABEL_POLICY = "error"


# ============================================================
# JSON-lines I/O
# ============================================================

def iter_json_records(path: Union[str, Path]) -> Iterable[Dict[str, Any]]:
    """
    Read newline-delimited JSON records.

    The .jsondk extension is treated as JSON Lines.
    """
    path = Path(path)

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"{path}:{line_number}: invalid JSON"
                ) from exc

            if not isinstance(obj, dict):
                raise ValueError(
                    f"{path}:{line_number}: expected a JSON object, "
                    f"got {type(obj).__name__}"
                )

            yield obj


# ============================================================
# Entity span validation / overlap handling
# ============================================================

def _normalise_entity(
    entity: Dict[str, Any],
    *,
    record_text: str,
    record_index: int,
) -> Optional[Dict[str, Any]]:
    """
    Validate and normalize one entity span.
    """
    if not isinstance(entity, dict):
        raise TypeError(
            f"Record {record_index}: entity must be a dict, "
            f"got {type(entity).__name__}"
        )

    if ENTITY_START_FIELD not in entity or ENTITY_END_FIELD not in entity:
        raise KeyError(
            f"Record {record_index}: entity is missing "
            f"{ENTITY_START_FIELD!r} or {ENTITY_END_FIELD!r}: {entity}"
        )

    start = int(entity[ENTITY_START_FIELD])
    end = int(entity[ENTITY_END_FIELD])

    if not (0 <= start <= end <= len(record_text)):
        raise ValueError(
            f"Record {record_index}: invalid span ({start}, {end}) "
            f"for text length {len(record_text)}."
        )

    label = entity.get(ENTITY_LABEL_FIELD)

    if label is None or str(label).strip() == "":
        if EMPTY_LABEL_POLICY == "skip":
            return None

        raise ValueError(
            f"Record {record_index}: entity ({start}, {end}) "
            f"has no {ENTITY_LABEL_FIELD!r}."
        )

    label = str(label).strip()
    observed_text = record_text[start:end]

    if VALIDATE_ENTITY_TEXT and ENTITY_TEXT_FIELD in entity:
        stored_text = entity.get(ENTITY_TEXT_FIELD)

        if stored_text is not None and str(stored_text) != observed_text:
            raise ValueError(
                f"Record {record_index}: entity text mismatch at "
                f"({start}, {end}). "
                f"record text gives {observed_text!r}, "
                f"entity field gives {stored_text!r}."
            )

    return {
        "start": start,
        "end": end,
        "text": observed_text,
        "label": label,
        "source": entity,
    }


def _resolve_overlaps(
    entities: List[Dict[str, Any]],
    *,
    record_index: int,
) -> List[Dict[str, Any]]:
    """
    Return non-overlapping entities according to OVERLAP_POLICY.
    """
    if not entities:
        return []

    if OVERLAP_POLICY == "longest":
        # Prefer longer spans first, then restore textual order.
        candidates = sorted(
            entities,
            key=lambda x: (
                -(x["end"] - x["start"]),
                x["start"],
                x["end"],
            ),
        )

        selected: List[Dict[str, Any]] = []

        for entity in candidates:
            overlaps = any(
                not (
                    entity["end"] <= other["start"]
                    or entity["start"] >= other["end"]
                )
                for other in selected
            )

            if not overlaps:
                selected.append(entity)

        return sorted(selected, key=lambda x: (x["start"], x["end"]))

    entities = sorted(
        entities,
        key=lambda x: (x["start"], x["end"])
    )

    resolved: List[Dict[str, Any]] = []

    for entity in entities:
        if not resolved:
            resolved.append(entity)
            continue

        previous = resolved[-1]

        if entity["start"] < previous["end"]:
            if OVERLAP_POLICY == "skip_later":
                continue

            if OVERLAP_POLICY == "error":
                raise ValueError(
                    f"Record {record_index}: overlapping entities: "
                    f"{previous['text']!r} "
                    f"({previous['start']}, {previous['end']}) and "
                    f"{entity['text']!r} "
                    f"({entity['start']}, {entity['end']})."
                )

            raise ValueError(
                f"Unsupported OVERLAP_POLICY={OVERLAP_POLICY!r}"
            )

        resolved.append(entity)

    return resolved


# ============================================================
# Inline ALF formatting
# ============================================================

def format_entity_inline(text: str, label: str) -> str:
    """
    Apply the adjustable inline format.

    Default:
        Peter Hansen + PERSON
        -> [PersonData]Peter Hansen[PERSON]
    """
    try:
        return INLINE_ENTITY_FORMAT.format(
            text=text,
            label=label,
        )
    except KeyError as exc:
        raise ValueError(
            "INLINE_ENTITY_FORMAT may only use "
            "{text} and {label} placeholders."
        ) from exc


def format_record_as_alf(
    record: Dict[str, Any],
    *,
    record_index: int = 0,
) -> Dict[str, Any]:
    """
    Convert a single .jsondk record into ALF inline text.

    Entity offsets are interpreted against record[TEXT_FIELD].
    """
    if TEXT_FIELD not in record:
        raise KeyError(
            f"Record {record_index}: missing text field "
            f"{TEXT_FIELD!r}."
        )

    record_text = record[TEXT_FIELD]

    if not isinstance(record_text, str):
        raise TypeError(
            f"Record {record_index}: {TEXT_FIELD!r} must be str, "
            f"got {type(record_text).__name__}."
        )

    raw_entities = record.get(ENTITIES_FIELD, [])

    if raw_entities is None:
        raw_entities = []

    if not isinstance(raw_entities, list):
        raise TypeError(
            f"Record {record_index}: {ENTITIES_FIELD!r} must be a list."
        )

    entities: List[Dict[str, Any]] = []

    for raw_entity in raw_entities:
        normalized = _normalise_entity(
            raw_entity,
            record_text=record_text,
            record_index=record_index,
        )

        if normalized is not None:
            entities.append(normalized)

    entities = _resolve_overlaps(
        entities,
        record_index=record_index,
    )

    # Build output left -> right so original whitespace and
    # punctuation outside entity spans remain unchanged.
    pieces: List[str] = []
    cursor = 0

    for entity in entities:
        start = entity["start"]
        end = entity["end"]

        pieces.append(record_text[cursor:start])

        pieces.append(
            format_entity_inline(
                entity["text"],
                entity["label"],
            )
        )

        cursor = end

    pieces.append(record_text[cursor:])

    alf_text = "".join(pieces)

    if KEEP_ORIGINAL_FIELDS:
        out = deepcopy(record)
        out[OUTPUT_TEXT_FIELD] = alf_text
        return out

    return {
        OUTPUT_TEXT_FIELD: alf_text
    }


# ============================================================
# File conversion
# ============================================================

def default_alfjson_path(
    input_path: Union[str, Path],
    output_dir: Optional[Union[str, Path]] = None,
) -> Path:
    """
    Example:
        NERset2_dev_aligned.jsondk
            ->
        NERset2_dev_aligned.alfjson
    """
    input_path = Path(input_path)

    if input_path.suffix.lower() != ".jsondk":
        raise ValueError(
            f"Expected a .jsondk input file, got: {input_path}"
        )

    output_name = input_path.with_suffix(".alfjson").name

    if output_dir is None:
        return input_path.with_suffix(".alfjson")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    return output_dir / output_name


def convert_jsondk_to_alfjson(
    input_path: Union[str, Path],
    output_path: Optional[Union[str, Path]] = None,
    *,
    output_dir: Optional[Union[str, Path]] = None,
) -> Dict[str, Any]:
    """
    Convert one .jsondk file to newline-delimited .alfjson.
    """
    input_path = Path(input_path)

    if input_path.suffix.lower() != ".jsondk":
        raise ValueError(
            f"Expected input extension .jsondk, got: {input_path}"
        )

    if not input_path.exists():
        raise FileNotFoundError(input_path)

    if output_path is not None and output_dir is not None:
        raise ValueError(
            "Use either output_path or output_dir, not both."
        )

    if output_path is None:
        output_path = default_alfjson_path(
            input_path,
            output_dir=output_dir,
        )
    else:
        output_path = Path(output_path)

        if output_path.suffix.lower() != ".alfjson":
            output_path = output_path.with_suffix(".alfjson")

        output_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

    n_records = 0
    n_entities = 0

    with Path(output_path).open(
        "w",
        encoding="utf-8",
    ) as fout:

        for record_index, record in enumerate(
            iter_json_records(input_path),
            start=1,
        ):
            raw_entities = record.get(
                ENTITIES_FIELD,
                [],
            ) or []

            converted = format_record_as_alf(
                record,
                record_index=record_index,
            )

            fout.write(
                json.dumps(
                    converted,
                    ensure_ascii=False,
                    separators=(",", ":"),
                )
                + "\n"
            )

            n_records += 1
            n_entities += len(raw_entities)

    return {
        "input": str(input_path),
        "output": str(output_path),
        "records": n_records,
        "entities_seen": n_entities,
        "inline_format": INLINE_ENTITY_FORMAT,
        "output_text_field": OUTPUT_TEXT_FIELD,
    }


# ============================================================
# Batch folder conversion
# ============================================================

def convert_jsondk_folder_to_alfjson(
    input_dir: Union[str, Path] = "danish_ner_datasets_aligned",
    output_dir: Optional[Union[str, Path]] = None,
    *,
    recursive: bool = False,
) -> Dict[str, Any]:
    """
    Convert all .jsondk files in a directory.

    If output_dir is omitted, each .alfjson file is written
    beside its corresponding .jsondk input.
    """
    input_dir = Path(input_dir)

    if not input_dir.exists():
        raise FileNotFoundError(input_dir)

    pattern = "**/*.jsondk" if recursive else "*.jsondk"
    input_files = sorted(input_dir.glob(pattern))

    if not input_files:
        raise FileNotFoundError(
            f"No .jsondk files found under {input_dir}"
        )

    results: List[Dict[str, Any]] = []

    for input_path in input_files:
        if output_dir is None:
            this_output_dir = None
        else:
            rel_parent = input_path.parent.relative_to(
                input_dir
            )
            this_output_dir = (
                Path(output_dir) / rel_parent
            )

        result = convert_jsondk_to_alfjson(
            input_path,
            output_dir=this_output_dir,
        )

        results.append(result)

    return {
        "input_dir": str(input_dir),
        "output_dir": (
            str(output_dir)
            if output_dir is not None
            else str(input_dir)
        ),
        "n_files": len(results),
        "n_records": sum(
            x["records"] for x in results
        ),
        "files": results,
        "inline_format": INLINE_ENTITY_FORMAT,
    }


# ============================================================
# Jupyter-friendly entry point
# ============================================================

def main(
    input_path: Union[str, Path] = "danish_ner_datasets_aligned",
    output_dir: Optional[Union[str, Path]] = None,
) -> Dict[str, Any]:
    """
    Jupyter usage
    -------------

    One file:

        from format_jsondk_to_alf import main

        result = main(
            "danish_ner_datasets_aligned/"
            "NERset2_dev_aligned.jsondk"
        )

    Whole folder:

        result = main(
            "danish_ner_datasets_aligned"
        )
    """
    input_path = Path(input_path)

    if input_path.is_file():
        return convert_jsondk_to_alfjson(
            input_path,
            output_dir=output_dir,
        )

    if input_path.is_dir():
        return convert_jsondk_folder_to_alfjson(
            input_dir=input_path,
            output_dir=output_dir,
        )

    raise FileNotFoundError(input_path)


if __name__ == "__main__":
    result = main()

    print(
        json.dumps(
            result,
            ensure_ascii=False,
            indent=2,
        )
    )


## Leave parent-label only

In [0]:
from __future__ import annotations

import json
from copy import deepcopy
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Union


# ============================================================
# USER-ADJUSTABLE SETTINGS
# ============================================================

# Input:  *.alfjson
# Output: *.alfjsonparent
#
# Inline parent-label format:
#     [PersonData]Peter Hansen[PERSON]
#
# Available placeholders:
#     {text}   = entity text
#     {label}  = Danish ALF parent label
PARENT_INLINE_FORMAT = "[PersonData]{text}[{label}]"

TEXT_FIELD = "text"
ENTITIES_FIELD = "alf_entities"
OUTPUT_TEXT_FIELD = "alf_text"

ENTITY_START_FIELD = "start"
ENTITY_END_FIELD = "end"
ENTITY_TEXT_FIELD = "text"
ENTITY_CHILD_LABEL_FIELD = "child_label"
ENTITY_PARENT_LABEL_FIELD = "parent_label"

# Default: create a compact output containing only the original
# text, parent-only ALF entities, and the parent-formatted text.
KEEP_ORIGINAL_FIELDS = False

# Metadata to retain when KEEP_ORIGINAL_FIELDS=False.
PASSTHROUGH_FIELDS = (
    "alf_dataset",
    "alf_split",
)

VALIDATE_ENTITY_TEXT = True

# "error", "skip_later", or "longest"
OVERLAP_POLICY = "error"

# "error" or "keep"
UNKNOWN_LABEL_POLICY = "error"


# ============================================================
# EXPLICIT DANISH CHILD -> DANISH PARENT PROJECTION
# ============================================================

DK_CHILD_TO_PARENT: Dict[str, str] = {
    # PER
    "PERSON": "PERSON",
    "PERSONALE": "PERSON",
    "PATIENT": "PERSON",

    # LOC
    "STED": "STED",
    "ADRESSE": "STED",
    "POSTNUMMER": "STED",

    # ORG
    "ORGANISATION": "ORGANISATION",
    "HOSPITAL": "ORGANISATION",

    # CONTACT
    "KONTAKT": "KONTAKT",
    "TELEFON": "KONTAKT",
    "EMAIL": "KONTAKT",

    # ID
    "ID": "ID",
    "CPR": "ID",

    # DATE
    "DATO": "DATO",
    "TID": "DATO",
    "VARIGHED": "DATO",

    # DEM
    "DEMOGRAFI": "DEMOGRAFI",
    "ETNICITET": "DEMOGRAFI",
    "RELIGION": "DEMOGRAFI",
    "POLITIK": "DEMOGRAFI",
    "SEKSUALITET": "DEMOGRAFI",
    "ALDER": "DEMOGRAFI",

    # HEALTH
    "SUNDHED": "SUNDHED",
    "DIAGNOSE": "SUNDHED",
    "MEDICIN": "SUNDHED",
    "TILSTAND": "SUNDHED",

    # REL
    "RELATION": "RELATION",

    # NUM
    "TAL": "TAL",
    "PROCENT": "TAL",
    "KARDINALTAL": "TAL",
    "BELØB": "TAL",

    # Defensive support for parent-only MISC.
    "MISC": "MISC",
}


def project_dk_child_to_parent(label: Optional[str]) -> Optional[str]:
    if label is None:
        return None

    label = str(label).strip()

    if label in DK_CHILD_TO_PARENT:
        return DK_CHILD_TO_PARENT[label]

    if UNKNOWN_LABEL_POLICY == "keep":
        return label

    raise KeyError(
        f"Unknown Danish ALF child label: {label!r}. "
        "Add it explicitly to DK_CHILD_TO_PARENT."
    )


def iter_alfjson(path: Union[str, Path]) -> Iterable[Dict[str, Any]]:
    path = Path(path)

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"{path}:{line_number}: invalid JSON"
                ) from exc

            if not isinstance(obj, dict):
                raise ValueError(
                    f"{path}:{line_number}: expected JSON object."
                )

            yield obj


def _normalise_entity(
    entity: Dict[str, Any],
    *,
    record_text: str,
    record_index: int,
) -> Dict[str, Any]:

    start = int(entity[ENTITY_START_FIELD])
    end = int(entity[ENTITY_END_FIELD])

    if not (0 <= start <= end <= len(record_text)):
        raise ValueError(
            f"Record {record_index}: invalid span ({start}, {end})."
        )

    if ENTITY_CHILD_LABEL_FIELD in entity:
        child_label = entity[ENTITY_CHILD_LABEL_FIELD]
    elif ENTITY_PARENT_LABEL_FIELD in entity:
        child_label = entity[ENTITY_PARENT_LABEL_FIELD]
    else:
        raise KeyError(
            f"Record {record_index}: entity has no "
            f"{ENTITY_CHILD_LABEL_FIELD!r} or "
            f"{ENTITY_PARENT_LABEL_FIELD!r}."
        )

    parent_label = project_dk_child_to_parent(child_label)
    observed_text = record_text[start:end]

    if VALIDATE_ENTITY_TEXT and ENTITY_TEXT_FIELD in entity:
        stored_text = entity.get(ENTITY_TEXT_FIELD)

        if stored_text is not None and str(stored_text) != observed_text:
            raise ValueError(
                f"Record {record_index}: entity text mismatch at "
                f"({start}, {end}): {stored_text!r} != {observed_text!r}"
            )

    return {
        "start": start,
        "end": end,
        "text": observed_text,
        "parent_label": parent_label,
    }


def _resolve_overlaps(
    entities: List[Dict[str, Any]],
    *,
    record_index: int,
) -> List[Dict[str, Any]]:

    if not entities:
        return []

    if OVERLAP_POLICY == "longest":
        candidates = sorted(
            entities,
            key=lambda x: (
                -(x["end"] - x["start"]),
                x["start"],
                x["end"],
            ),
        )

        selected: List[Dict[str, Any]] = []

        for entity in candidates:
            overlaps = any(
                not (
                    entity["end"] <= other["start"]
                    or entity["start"] >= other["end"]
                )
                for other in selected
            )

            if not overlaps:
                selected.append(entity)

        return sorted(
            selected,
            key=lambda x: (x["start"], x["end"]),
        )

    entities = sorted(
        entities,
        key=lambda x: (x["start"], x["end"]),
    )

    resolved: List[Dict[str, Any]] = []

    for entity in entities:
        if not resolved:
            resolved.append(entity)
            continue

        previous = resolved[-1]

        if entity["start"] < previous["end"]:
            if OVERLAP_POLICY == "skip_later":
                continue

            if OVERLAP_POLICY == "error":
                raise ValueError(
                    f"Record {record_index}: overlapping entities "
                    f"{previous!r} and {entity!r}"
                )

            raise ValueError(
                f"Unsupported OVERLAP_POLICY={OVERLAP_POLICY!r}"
            )

        resolved.append(entity)

    return resolved


def format_parent_entity_inline(text: str, parent_label: str) -> str:
    return PARENT_INLINE_FORMAT.format(
        text=text,
        label=parent_label,
    )


def build_parent_alf_text(
    record_text: str,
    entities: List[Dict[str, Any]],
) -> str:

    pieces: List[str] = []
    cursor = 0

    for entity in entities:
        start = entity["start"]
        end = entity["end"]

        pieces.append(record_text[cursor:start])
        pieces.append(
            format_parent_entity_inline(
                entity["text"],
                entity["parent_label"],
            )
        )

        cursor = end

    pieces.append(record_text[cursor:])

    return "".join(pieces)


def convert_record_to_parent_labels(
    record: Dict[str, Any],
    *,
    record_index: int = 0,
) -> Dict[str, Any]:

    if TEXT_FIELD not in record:
        raise KeyError(
            f"Record {record_index}: missing {TEXT_FIELD!r}."
        )

    record_text = record[TEXT_FIELD]

    if not isinstance(record_text, str):
        raise TypeError(
            f"Record {record_index}: {TEXT_FIELD!r} must be str."
        )

    raw_entities = record.get(ENTITIES_FIELD, []) or []

    if not isinstance(raw_entities, list):
        raise TypeError(
            f"Record {record_index}: {ENTITIES_FIELD!r} must be list."
        )

    parent_entities = [
        _normalise_entity(
            entity,
            record_text=record_text,
            record_index=record_index,
        )
        for entity in raw_entities
    ]

    parent_entities = _resolve_overlaps(
        parent_entities,
        record_index=record_index,
    )

    parent_text = build_parent_alf_text(
        record_text,
        parent_entities,
    )

    if KEEP_ORIGINAL_FIELDS:
        out = deepcopy(record)
        out[ENTITIES_FIELD] = parent_entities
        out[OUTPUT_TEXT_FIELD] = parent_text
        out["alf_label_level"] = "parent"
        out["alf_label_language"] = "da"

        # Remove old ALF child/English token-label fields.
        for field in (
            "alf_ner_tags",
            "alf_ner_tags_en",
            "alf_mbert_token_classes",
            "alf_mbert_token_classes_en",
        ):
            out.pop(field, None)

        return out

    out: Dict[str, Any] = {
        TEXT_FIELD: record_text,
        ENTITIES_FIELD: parent_entities,
        OUTPUT_TEXT_FIELD: parent_text,
        "alf_label_level": "parent",
        "alf_label_language": "da",
    }

    for field in PASSTHROUGH_FIELDS:
        if field in record:
            out[field] = deepcopy(record[field])

    return out


def default_parent_path(
    input_path: Union[str, Path],
    output_dir: Optional[Union[str, Path]] = None,
) -> Path:

    input_path = Path(input_path)

    if input_path.suffix.lower() != ".alfjson":
        raise ValueError(
            f"Expected .alfjson input, got: {input_path}"
        )

    output_name = input_path.with_suffix(".alfjsonparent").name

    if output_dir is None:
        return input_path.with_suffix(".alfjsonparent")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    return output_dir / output_name


def convert_alfjson_to_parent(
    input_path: Union[str, Path],
    output_path: Optional[Union[str, Path]] = None,
    *,
    output_dir: Optional[Union[str, Path]] = None,
) -> Dict[str, Any]:

    input_path = Path(input_path)

    if input_path.suffix.lower() != ".alfjson":
        raise ValueError(
            f"Expected .alfjson input, got: {input_path}"
        )

    if not input_path.exists():
        raise FileNotFoundError(input_path)

    if output_path is not None and output_dir is not None:
        raise ValueError(
            "Use output_path or output_dir, not both."
        )

    if output_path is None:
        output_path = default_parent_path(
            input_path,
            output_dir=output_dir,
        )
    else:
        output_path = Path(output_path)

        if output_path.suffix.lower() != ".alfjsonparent":
            output_path = Path(str(output_path) + ".alfjsonparent")

        output_path.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

    n_records = 0
    n_entities = 0

    with Path(output_path).open("w", encoding="utf-8") as fout:
        for record_index, record in enumerate(
            iter_alfjson(input_path),
            start=1,
        ):
            converted = convert_record_to_parent_labels(
                record,
                record_index=record_index,
            )

            fout.write(
                json.dumps(
                    converted,
                    ensure_ascii=False,
                    separators=(",", ":"),
                )
                + "\n"
            )

            n_records += 1
            n_entities += len(
                converted.get(ENTITIES_FIELD, [])
            )

    return {
        "input": str(input_path),
        "output": str(output_path),
        "records": n_records,
        "entities": n_entities,
        "label_level": "parent",
        "label_language": "da",
        "inline_format": PARENT_INLINE_FORMAT,
    }


def convert_alfjson_folder_to_parent(
    input_dir: Union[str, Path] = "danish_ner_datasets_aligned",
    output_dir: Optional[Union[str, Path]] = None,
    *,
    recursive: bool = False,
) -> Dict[str, Any]:

    input_dir = Path(input_dir)

    if not input_dir.exists():
        raise FileNotFoundError(input_dir)

    pattern = "**/*.alfjson" if recursive else "*.alfjson"
    input_files = sorted(input_dir.glob(pattern))

    if not input_files:
        raise FileNotFoundError(
            f"No .alfjson files found under {input_dir}"
        )

    results: List[Dict[str, Any]] = []

    for input_path in input_files:
        if output_dir is None:
            this_output_dir = None
        else:
            rel_parent = input_path.parent.relative_to(input_dir)
            this_output_dir = Path(output_dir) / rel_parent

        result = convert_alfjson_to_parent(
            input_path,
            output_dir=this_output_dir,
        )

        results.append(result)

    return {
        "input_dir": str(input_dir),
        "output_dir": (
            str(output_dir)
            if output_dir is not None
            else str(input_dir)
        ),
        "n_files": len(results),
        "n_records": sum(x["records"] for x in results),
        "n_entities": sum(x["entities"] for x in results),
        "files": results,
        "label_level": "parent",
        "label_language": "da",
    }


def main(
    input_path: Union[str, Path] = "danish_ner_datasets_aligned",
    output_dir: Optional[Union[str, Path]] = None,
) -> Dict[str, Any]:
    """
    Jupyter-friendly entry point.

    One file:
        result = main(".../NERset2_dev_aligned.alfjson")

    Whole folder:
        result = main("danish_ner_datasets_aligned")
    """
    input_path = Path(input_path)

    if input_path.is_file():
        return convert_alfjson_to_parent(
            input_path,
            output_dir=output_dir,
        )

    if input_path.is_dir():
        return convert_alfjson_folder_to_parent(
            input_dir=input_path,
            output_dir=output_dir,
        )

    raise FileNotFoundError(input_path)


if __name__ == "__main__":
    result = main()

    print(
        json.dumps(
            result,
            ensure_ascii=False,
            indent=2,
        )
    )


# sample runnable code
optimizer muon/adamw
device cuda/mps

FacebookAI/xlm-roberta-large
KennethEnevoldsen/dfm-sentence-encoder-large-exp2-no-lang-align
intfloat/multilingual-e5-large-instruct
intfloat/multilingual-e5-large
icrosoft/harrier-oss-v1-0.6b


In [0]:
# %pip install torch transformers datasets requests

In [0]:
# XLM-R large
%run mixed_public_clinical_ner_demo_seed42_cacheclean.py \
    --task core \
    --model-name FacebookAI/xlm-roberta-large \
    --optimizer adamw \
    --device mps \
    --public-per-split 1 \
    --epochs 1

In [0]:
# XLM-R large muon
%run mixed_public_clinical_ner_demo_seed42_cacheclean.py \
    --task core \
    --model-name FacebookAI/xlm-roberta-large \
    --optimizer muon \
    --device mps \
    --public-per-split 1 \
    --epochs 1

In [0]:
# Danish sentence encoder
%run mixed_public_clinical_ner_demo_seed42_cacheclean.py \
    --task core \
    --model-name KennethEnevoldsen/dfm-sentence-encoder-large-exp2-no-lang-align \
    --optimizer adamw \
    --device mps \
    --public-per-split 1 \
    --epochs 1

In [0]:
# !multilingual-e5-large
%run mixed_public_clinical_ner_demo_seed42_cacheclean.py \
    --task core \
    --model-name intfloat/multilingual-e5-large \
    --optimizer adamw \
    --device mps \
    --public-per-split 1 \
    --epochs 1

In [0]:
# multilingual-e5-large-instruct
%run mixed_public_clinical_ner_demo_seed42_cacheclean.py \
    --task core \
    --model-name intfloat/multilingual-e5-large-instruct \
    --instruction "Identify person, location, and organization named entities in Danish psychotherapy text." \
    --optimizer adamw \
    --public-per-split 1 \
    --epochs 1 \
    --device mps

In [0]:
# !harrier
%run mixed_public_clinical_ner_demo_seed42_cacheclean.py \
    --task core \
    --model-name microsoft/harrier-oss-v1-0.6b \
    --optimizer muon \
    --device mps \
    --public-per-split 1 \
    --epochs 1

In [0]:
# extended labels (no per/loc/org)
%run mixed_public_clinical_ner_demo_seed42_cacheclean.py \
    --task extended \
    --model-name microsoft/harrier-oss-v1-0.6b \
    --optimizer adamw \
    --public-per-split 1 \
    --epochs 1 \
    --device mps